<a href="https://colab.research.google.com/github/Cousar/Colab-codes-for-TE-project/blob/main/Pickup_Charges.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# from connect import *

# exec(open(r"S:\NHC\Scripts\ChargeNew\ChargeScorerRefined.py").read())

#============================================================
# IMPORTS
# ============================================================

import os
import re
import json
import numpy as np
from connect import get_current


# ============================================================
# CONFIGURATION
# ============================================================

OUTPUT_BASE_DIR = "S:/NHC/Scripts/ChargeScores/30mmR"

CHARGE_EVALUATION_PREFIX = "charge"
BRASS_MATERIAL_KEYWORD = "brass"

EXPORT_NPY = True
EXPORT_TXT = True
EXPORT_METADATA_JSON = True


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def sanitize_for_filename(text):
    text = str(text)
    text = text.strip()
    text = text.replace(" ", "")
    text = text.replace("¤", "")
    return re.sub(r"[^A-Za-z0-9_\-\.\+]", "_", text)


def extract_energy_from_plan_name(plan_name):
    match = re.search(r"(\d+)\s*MeV", plan_name, re.IGNORECASE)
    if match:
        return "{}MeV".format(match.group(1))
    return "UnknownEnergy"


def extract_shift_from_plan_name(plan_name):
    match = re.search(r"Shift\s*=\s*([+\-]?\d+(?:\.\d+)?)\s*mm", plan_name, re.IGNORECASE)
    if match:
        shift_value = match.group(1)
        return "{}mm".format(shift_value)
    return "UnknownShift"


def numpy_to_float(value):
    return float(np.asarray(value))


# ============================================================
# RAYSTATION OBJECTS
# ============================================================

patient = get_current("Patient")
case = get_current("Case")

if patient is None or case is None:
    raise Exception("No active patient or case found in RayStation.")

print("========================================")
print("Patient Name : {}".format(patient.Name))
print("Patient ID   : {}".format(patient.PatientID))
print("Case Name    : {}".format(case.CaseName))
print("========================================")


# ============================================================
# FIND CHARGE DOSE EVALUATIONS
# ============================================================

charge_evaluations = []

for fe_index, fraction_evaluation in enumerate(case.TreatmentDelivery.FractionEvaluations):

    for doe_index, dose_on_examination in enumerate(fraction_evaluation.DoseOnExaminations):

        for ev_index, evaluation in enumerate(dose_on_examination.DoseEvaluations):

            if evaluation is None:
                continue

            evaluation_name = evaluation.Name or ""
            evaluation_name = evaluation_name.strip()

            if not evaluation_name.lower().startswith(CHARGE_EVALUATION_PREFIX.lower()):
                continue

            charge_evaluations.append({
                "fraction_evaluation_index": fe_index,
                "dose_on_examination_index": doe_index,
                "dose_evaluation_index": ev_index,
                "evaluation": evaluation,
                "evaluation_name": evaluation_name
            })


if len(charge_evaluations) == 0:
    raise Exception("No charge DoseEvaluations found with names starting with 'Charge'.")

print("\n========================================")
print("Found {} charge evaluation(s):".format(len(charge_evaluations)))

for idx, item in enumerate(charge_evaluations):
    print("  [{}] {}".format(idx, item["evaluation_name"]))

print("========================================\n")


# ============================================================
# FIND ROIS WITH BRASS MATERIAL
# ============================================================

brass_rois = []

for roi in case.PatientModel.RegionsOfInterest:

    roi_name = roi.Name

    try:
        material_name = roi.RoiMaterial.OfMaterial.Name
    except Exception:
        continue

    if material_name is None:
        continue

    if BRASS_MATERIAL_KEYWORD.lower() not in material_name.lower():
        continue

    brass_rois.append({
        "roi_name": roi_name,
        "material_name": material_name
    })


if len(brass_rois) == 0:
    raise Exception("No ROIs with Brass material found.")


print("\n========================================")
print("Found {} Brass ROI(s):".format(len(brass_rois)))

for idx, item in enumerate(brass_rois):
    print("  [{}] ROI: {} | Material: {}".format(
        idx,
        item["roi_name"],
        item["material_name"]
    ))

print("========================================\n")


# ============================================================
# PROCESS CHARGE EVALUATIONS
# ============================================================

for item in charge_evaluations:

    evaluation = item["evaluation"]
    evaluation_name = item["evaluation_name"]
    ev_index = item["dose_evaluation_index"]

    print("\n========================================")
    print("Processing charge evaluation:")
    print("  DoseEvaluation index : {}".format(ev_index))
    print("  DoseEvaluation name  : {}".format(evaluation_name))

    # ------------------------------------------------------------
    # Get corresponding TreatmentPlan by index
    # Assumption:
    # DoseEvaluations[i] corresponds to TreatmentPlans[i]
    # ------------------------------------------------------------

    try:
        corresponding_plan = case.TreatmentPlans[ev_index]
        corresponding_plan_name = corresponding_plan.Name
    except Exception as e:
        print("  -> Could not find corresponding TreatmentPlan for DoseEvaluation[{}]. Error: {}".format(
            ev_index, e
        ))
        continue

    print("  TreatmentPlan name   : {}".format(corresponding_plan_name))

    # ------------------------------------------------------------
    # Extract energy and shift from TreatmentPlan name
    # ------------------------------------------------------------

    energy_folder = extract_energy_from_plan_name(corresponding_plan_name)
    shift_folder = extract_shift_from_plan_name(corresponding_plan_name)

    output_dir = os.path.join(
        OUTPUT_BASE_DIR,
        energy_folder,
        shift_folder
    )

    os.makedirs(output_dir, exist_ok=True)

    print("  Energy folder        : {}".format(energy_folder))
    print("  Shift folder         : {}".format(shift_folder))
    print("  Output folder        : {}".format(output_dir))

    # ------------------------------------------------------------
    # Extract charge grid
    # ------------------------------------------------------------

    try:
        charge_grid = np.array(evaluation.DoseValues.DoseData)
    except Exception as e:
        print("  -> Failed to read DoseData. Error: {}".format(e))
        continue

    if charge_grid is None or charge_grid.size == 0:
        print("  -> Empty charge grid. Skipped.")
        continue

    charge_grid_flat = charge_grid.ravel()

    print("  Charge grid shape    : {}".format(charge_grid.shape))
    print("  Number of voxels     : {}".format(charge_grid_flat.size))

    # ------------------------------------------------------------
    # Whole-grid metrics
    # ------------------------------------------------------------

    max_charge_grid = np.max(charge_grid_flat)
    sum_charge_grid = np.sum(charge_grid_flat)

    print("  Whole grid max charge: {}".format(max_charge_grid))
    print("  Whole grid sum charge: {}".format(sum_charge_grid))

    # ------------------------------------------------------------
    # Brass ROI metrics
    # Important:
    # Voxel indices are now taken from the corresponding plan.
    # ------------------------------------------------------------

    roi_results = []

    try:
        fraction_dose = corresponding_plan.BeamSets[0].FractionDose
    except Exception as e:
        print("  -> Could not access FractionDose of corresponding plan. Error: {}".format(e))
        continue

    for roi_item in brass_rois:

        roi_name = roi_item["roi_name"]
        material_name = roi_item["material_name"]

        print("  Getting voxel indices for ROI: {}".format(roi_name))

        try:
            dose_grid_roi = fraction_dose.GetDoseGridRoi(RoiName=roi_name)
            voxel_indices = dose_grid_roi.RoiVolumeDistribution.VoxelIndices

        except Exception as e:
            print("    -> Failed to get voxel indices for ROI '{}'. Error: {}".format(
                roi_name, e
            ))
            continue

        if voxel_indices is None or len(voxel_indices) == 0:
            print("    -> No voxel indices found for ROI: {}".format(roi_name))
            continue

        charge_values_roi = charge_grid_flat[voxel_indices]

        max_charge_roi = np.max(charge_values_roi)
        sum_charge_roi = np.sum(charge_values_roi)

        roi_result = {
            "roi_name": roi_name,
            "material_name": material_name,
            "number_of_voxels": len(voxel_indices),
            "max_charge": max_charge_roi,
            "sum_charge": sum_charge_roi
        }

        roi_results.append(roi_result)

        print("    ROI: {}".format(roi_name))
        print("      Material        : {}".format(material_name))
        print("      Number of voxels: {}".format(len(voxel_indices)))
        print("      Max charge      : {}".format(max_charge_roi))
        print("      Sum charge      : {}".format(sum_charge_roi))

    # ------------------------------------------------------------
    # Store calculated results
    # ------------------------------------------------------------

    item["corresponding_plan_name"] = corresponding_plan_name
    item["energy_folder"] = energy_folder
    item["shift_folder"] = shift_folder
    item["output_dir"] = output_dir
    item["charge_grid"] = charge_grid
    item["charge_grid_flat"] = charge_grid_flat
    item["max_charge_grid"] = max_charge_grid
    item["sum_charge_grid"] = sum_charge_grid
    item["roi_results"] = roi_results

    # ------------------------------------------------------------
    # Export results
    # ------------------------------------------------------------

    safe_plan_name = sanitize_for_filename(corresponding_plan_name)

    # ---- TXT summary ----

    if EXPORT_TXT:

        txt_file_path = os.path.join(
            output_dir,
            "{}_charge_summary.txt".format(safe_plan_name)
        )

        with open(txt_file_path, "w") as f:

            f.write("PatientName\t{}\n".format(patient.Name))
            f.write("PatientID\t{}\n".format(patient.PatientID))
            f.write("CaseName\t{}\n".format(case.CaseName))
            f.write("TreatmentPlanName\t{}\n".format(corresponding_plan_name))
            f.write("DoseEvaluationName\t{}\n".format(evaluation_name))
            f.write("DoseEvaluationIndex\t{}\n".format(ev_index))
            f.write("EnergyFolder\t{}\n".format(energy_folder))
            f.write("ShiftFolder\t{}\n".format(shift_folder))
            f.write("\n")

            f.write("Metric\tValue\n")
            f.write("Max_Charge_WholeGrid\t{}\n".format(max_charge_grid))
            f.write("Sum_Charge_WholeGrid\t{}\n".format(sum_charge_grid))

            for roi_result in roi_results:
                roi_name_safe = sanitize_for_filename(roi_result["roi_name"])

                f.write("Max_Charge_ROI_{}\t{}\n".format(
                    roi_name_safe,
                    roi_result["max_charge"]
                ))

                f.write("Sum_Charge_ROI_{}\t{}\n".format(
                    roi_name_safe,
                    roi_result["sum_charge"]
                ))

                f.write("NumberOfVoxels_ROI_{}\t{}\n".format(
                    roi_name_safe,
                    roi_result["number_of_voxels"]
                ))

        print("  -> TXT summary exported: {}".format(txt_file_path))

    # ---- JSON metadata/results ----

    if EXPORT_METADATA_JSON:

        json_file_path = os.path.join(
            output_dir,
            "{}_charge_summary.json".format(safe_plan_name)
        )

        json_data = {
            "patient_name": str(patient.Name),
            "patient_id": str(patient.PatientID),
            "case_name": str(case.CaseName),
            "treatment_plan_name": str(corresponding_plan_name),
            "dose_evaluation_name": str(evaluation_name),
            "dose_evaluation_index": int(ev_index),
            "energy_folder": str(energy_folder),
            "shift_folder": str(shift_folder),
            "charge_grid_shape": list(charge_grid.shape),
            "number_of_voxels_whole_grid": int(charge_grid_flat.size),
            "max_charge_whole_grid": numpy_to_float(max_charge_grid),
            "sum_charge_whole_grid": numpy_to_float(sum_charge_grid),
            "roi_results": []
        }

        for roi_result in roi_results:
            json_data["roi_results"].append({
                "roi_name": str(roi_result["roi_name"]),
                "material_name": str(roi_result["material_name"]),
                "number_of_voxels": int(roi_result["number_of_voxels"]),
                "max_charge": numpy_to_float(roi_result["max_charge"]),
                "sum_charge": numpy_to_float(roi_result["sum_charge"])
            })

        with open(json_file_path, "w") as f:
            json.dump(json_data, f, indent=4)

        print("  -> JSON summary exported: {}".format(json_file_path))

    # ---- NumPy arrays ----

    if EXPORT_NPY:

        npy_3d_path = os.path.join(
            output_dir,
            "{}_charge_grid_3D.npy".format(safe_plan_name)
        )

        npy_flat_path = os.path.join(
            output_dir,
            "{}_charge_grid_flat.npy".format(safe_plan_name)
        )

        np.save(npy_3d_path, charge_grid)
        np.save(npy_flat_path, charge_grid_flat)

        print("  -> 3D charge grid exported: {}".format(npy_3d_path))
        print("  -> Flat charge grid exported: {}".format(npy_flat_path))


print("\n========================================")
print("Charge export completed.")
print("========================================")